# Heart — XAI extension

The reusable XAI framework is applied to the pre-specified L2 logistic-regression model with 10-fold cross-validation, using the stratified 50/50 split.

Standardisation is fitted on the training partition before internal cross-validation; it is not refitted within each fold.

The framework generates explanations and assesses bootstrap-refit stability. The ordinary global permutation analysis uses ROC AUC, while stability uses negative log loss consistently across the four retrainable datasets. Results are exported to `../../results/heart`.



The exported global SHAP analysis uses SHAP 0.51.0 `PermutationExplainer`.


## Data and model preparation


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import shap
import sklearn
import statsmodels
from sklearn.base import clone
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "xai_extension":
    raise RuntimeError(
        "Run this notebook with the working directory set to "
        "thesis/src/xai_extension. Current directory: "
        f"{NOTEBOOK_DIR}"
    )

PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
FRAMEWORK_DIR = PROJECT_ROOT / "src" / "xai_framework"
if str(FRAMEWORK_DIR) not in sys.path:
    sys.path.insert(0, str(FRAMEWORK_DIR))

from run_pipeline import run_pipeline

file_path = "../../data/heart/processed.cleveland.data"

# Define the source dataset columns.
column_names = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
    'restecg', 'thalach', 'exang', 'oldpeak',
    'slope', 'ca', 'thal', 'target'
]

df = pd.read_csv(file_path, names=column_names)

# Replace the source missing-value marker.
df.replace('?', np.nan, inplace=True)

# Convert the source columns to numeric values.
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Remove incomplete observations.
df = df.dropna()

# Convert the diagnosis to a binary outcome.
df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)

X = df.drop('target', axis=1)
y = df['target']

feature_names = X.columns.tolist()

# The source paper uses equal training and test subsets. Split before
# learning preprocessing parameters to prevent test-data leakage.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=0.50,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train = pd.DataFrame(
    scaler.fit_transform(X_train_raw),
    columns=feature_names,
    index=X_train_raw.index,
)
X_test = pd.DataFrame(
    scaler.transform(X_test_raw),
    columns=feature_names,
    index=X_test_raw.index,
)


c:\Users\catia\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
lr_10fold = LogisticRegressionCV(Cs=10, cv=10, penalty='l2', solver='liblinear', max_iter=1000, random_state=42)
lr_10fold.fit(X_train, y_train)


LogisticRegressionCV(cv=10, max_iter=1000, random_state=42, solver='liblinear')

In [3]:
# Pre-specified XAI model: the 10-fold L2 logistic-regression variant.
# It is chosen from the replicated study design, not by inspecting the
# held-out test ranking above.
final_model = lr_10fold
selected_model_name = "Logistic Regression (10-fold CV)"


In [4]:
best_model = final_model

print("Final XAI model:", type(best_model).__name__)
print("Pre-specified model:", selected_model_name)
print("Selection did not use held-out test performance.")


Final XAI model: LogisticRegressionCV
Pre-specified model: Logistic Regression (10-fold CV)
Selection did not use held-out test performance.


In [5]:


def train_model_fn(X_bootstrap, y_bootstrap):
    """Refit the fixed Heart estimator on one bootstrap training sample."""
    model = clone(best_model)
    model.fit(X_bootstrap, y_bootstrap)
    return model


## Execute the XAI pipeline


In [6]:
PILOT = False

output_dir = (
    PROJECT_ROOT / "results" / "pilot_stability" / "heart"
    if PILOT
    else PROJECT_ROOT / "results" / "heart"
)

config = {
    "dataset_name": "heart",
    "model": best_model,

    "X_train": X_train,
    "X_test": X_test,

    "y_train": y_train,
    "y_test": y_test,

    "feature_names": feature_names,

    # Repeated local explanations
    "local_cases_per_type": 2 if PILOT else 10,
    "lime_seeds": list(range(2 if PILOT else 20)),
    "lime_num_features": 5,
    "lime_stability_top_k": 3,

    "train_model_fn": train_model_fn,

    # Bootstrap-refit stability
    "stability_n_bootstrap": 10 if PILOT else 100,
    "stability_bootstrap_seed": 42,
    "stability_scoring": "neg_log_loss",
    "stability_n_repeats": 10,
    "stability_permutation_seed": 42,
    "stability_top_k": 3,

    # Ordinary global permutation analysis
    "scoring": "roc_auc",

    # Pilot and final outputs are kept separate
    "output_dir": str(output_dir),
}

print("Working directory:", NOTEBOOK_DIR)
print("Framework directory:", FRAMEWORK_DIR)
print("Pilot mode:", PILOT)
print("Number of bootstrap refits:", config["stability_n_bootstrap"])
print("Output directory:", output_dir)

experiment = run_pipeline(config)


Working directory: C:\Users\catia\Desktop\Blackboard\AASB\github\thesis\src\xai_extension
Framework directory: C:\Users\catia\Desktop\Blackboard\AASB\github\thesis\src\xai_framework
Pilot mode: False
Number of bootstrap refits: 100
Output directory: C:\Users\catia\Desktop\Blackboard\AASB\github\thesis\results\heart


PermutationExplainer explainer: 150it [00:39,  3.73it/s]                         


Results exported to: C:\Users\catia\Desktop\Blackboard\AASB\github\thesis\results\heart


## XAI framework output sanity checks


In [7]:
print("Dataset:", config["dataset_name"])
print("Model:", type(config["model"]).__name__)

print("X_train shape:", config["X_train"].shape)
print("X_test shape:", config["X_test"].shape)

print("y_train length:", len(config["y_train"]))
print("y_test length:", len(config["y_test"]))

print("Feature names:", len(config["feature_names"]))


Dataset: heart
Model: LogisticRegressionCV
X_train shape: (148, 13)
X_test shape: (149, 13)
y_train length: 148
y_test length: 149
Feature names: 13


### Input consistency


In [8]:
assert config["X_train"].shape[1] == len(config["feature_names"])
assert config["X_test"].shape[1] == len(config["feature_names"])

assert len(config["X_train"]) == len(config["y_train"])
assert len(config["X_test"]) == len(config["y_test"])

print("Basic shape validation: OK")


Basic shape validation: OK


### Prediction-interface check


In [9]:
model = config["model"]
X_test = config["X_test"]

if hasattr(model, "predict_proba"):
    probs = model.predict_proba(X_test)[:, 1]
    print("Probability range:", probs.min(), probs.max())

preds = model.predict(X_test)

print("Predictions:", np.unique(preds, return_counts=True))
print("Prediction test: OK")


Probability range: 0.26801784968312586 0.7620684993860655
Predictions: (array([0, 1]), array([80, 69]))
Prediction test: OK


### Framework results


In [10]:
global_results = experiment.state.results["global"]
local_results = experiment.state.results["local"]
stability_results = experiment.state.results["stability"]

print(global_results.keys())
print(local_results.keys())

if stability_results is not None:
    print(stability_results.keys())
else:
    print("Stability: not available")


dict_keys(['native_importance', 'permutation_importance', 'shap_importance', 'shap_values', 'shap_raw_values', 'explainer', 'shap_explainer_class', 'rank_compare', 'rank_agreement'])
dict_keys(['cases', 'lime', 'lime_summary', 'lime_seed_stability'])
dict_keys(['raw', 'summary', 'rank_stability'])


### Global explanations


In [11]:
native_df = global_results["native_importance"]
perm_df = global_results["permutation_importance"]
shap_df = global_results["shap_importance"]

print("Native:")
display(native_df.head())

print("Permutation:")
display(perm_df.head())

print("SHAP:")
display(shap_df.head())


Native:


,Feature,Importance,RawValue,Method
12,thal,0.153951,0.153951,coefficients
2,cp,0.130170,0.130170,coefficients
8,exang,0.114381,0.114381,coefficients
7,thalach,0.104741,-0.104741,coefficients
11,ca,0.100806,0.100806,coefficients


Permutation:


,Feature,Importance,Std
12,thal,0.035833,0.013175
11,ca,0.029493,0.008113
1,sex,0.014293,0.005012
7,thalach,0.011286,0.006377
2,cp,0.010707,0.008875


SHAP:


,Feature,MeanAbsSHAP,MeanSHAP
0,thal,0.036137,-0.002527
1,cp,0.026465,-0.001573
2,exang,0.025664,-0.001055
3,thalach,0.024347,0.005821
4,slope,0.022207,0.000472


### Feature coverage


In [12]:
if native_df is not None:
    print("Native features:", len(native_df))

print("Permutation features:", len(perm_df))
print("SHAP features:", len(shap_df))
print("Expected features:", len(config["feature_names"]))


Native features: 13
Permutation features: 13
SHAP features: 13
Expected features: 13


### Numerical integrity


In [13]:
print("Permutation NaNs:", perm_df.isna().sum().sum())
print("SHAP NaNs:", shap_df.isna().sum().sum())

if native_df is not None:
    print("Native NaNs:", native_df.isna().sum().sum())


Permutation NaNs: 0
SHAP NaNs: 0
Native NaNs: 0


### Rank agreement


In [14]:
print(global_results["rank_agreement"])

display(global_results["rank_compare"].head())


{'Native_vs_SHAP': np.float64(0.9780219780219779), 'Native_vs_Permutation': np.float64(0.6208791208791209), 'SHAP_vs_Permutation': np.float64(0.6153846153846153)}


,Feature,Importance_x,RawValue,Method,NativeRank,MeanAbsSHAP,MeanSHAP,SHAPRank,Importance_y,Std,PermutationRank
0,thal,0.153951,0.153951,coefficients,1.0,0.036137,-0.002527,1.0,0.035833,0.013175,1.0
1,cp,0.130170,0.130170,coefficients,2.0,0.026465,-0.001573,2.0,0.010707,0.008875,5.0
2,exang,0.114381,0.114381,coefficients,3.0,0.025664,-0.001055,3.0,0.005888,0.007579,7.0
3,thalach,0.104741,-0.104741,coefficients,4.0,0.024347,0.005821,4.0,0.011286,0.006377,4.0
4,ca,0.100806,0.100806,coefficients,5.0,0.022150,0.001594,6.0,0.029493,0.008113,2.0


### Representative cases


In [15]:
cases = local_results["cases"]

display(
    cases[
        ["CaseType", "y_true", "y_pred", "y_prob"]
    ]
)


,CaseType,y_true,y_pred,y_prob
84,TP,1,1,0.762068
2,TP,1,1,0.756083
125,TP,1,1,0.745467
14,TP,1,1,0.736840
46,TP,1,1,0.735711
26,TP,1,1,0.728936
18,TP,1,1,0.723356
74,TP,1,1,0.717047
11,TP,1,1,0.716343
60,TP,1,1,0.707111


### Local explanations and seed stability


In [16]:
lime_df = local_results["lime"]
lime_summary_df = local_results["lime_summary"]
lime_seed_stability_df = local_results["lime_seed_stability"]

print("Repeated LIME rows:", len(lime_df))
print("Cases by category:")
display(local_results["cases"].groupby("CaseType").size().rename("N cases"))
print("Seeds per case:")
display(lime_df.groupby(["CaseType", "CaseIndex"])["LIMESeed"].nunique().describe())
print("Aggregated LIME summary:")
display(lime_summary_df.head(20))
print("LIME seed-stability summary:")
display(
    lime_seed_stability_df.groupby("CaseType")[[
        "MeanPairwiseSpearman", "MeanTopKOverlap",
        "MeanRuleWeightSD", "MeanRuleSignConsistency"
    ]].mean().round(3)
)


Repeated LIME rows: 5000
Cases by category:


CaseType
Borderline    10
FN            10
FP            10
TN            10
TP            10
Name: N cases, dtype: int64

Seeds per case:


count    50.0
mean     20.0
std       0.0
min      20.0
25%      20.0
50%      20.0
75%      20.0
max      20.0
Name: LIMESeed, dtype: float64

Aggregated LIME summary:


,CaseType,Rule,MeanWeight,StdWeight,MeanAbsWeight,MedianWeight,PositiveWeightFraction,NExplanations,NCases,NSeeds,SelectionFrequency
0,Borderline,cp <= -0.21,-0.060488,0.001168,0.060488,-0.060457,0.0,120,6,20,0.600
1,Borderline,0.32 < cp <= 0.85,0.060430,0.001066,0.060430,0.060313,1.0,80,4,20,0.400
2,Borderline,thalach <= -0.52,0.048558,0.001077,0.048558,0.048520,1.0,40,2,20,0.200
3,Borderline,ca > 0.40,0.040292,0.001357,0.040292,0.040376,1.0,20,1,20,0.100
4,Borderline,-0.70 < exang <= 1.42,0.039582,0.001280,0.039582,0.039633,1.0,40,2,20,0.200
5,Borderline,exang <= -0.70,-0.039314,0.001257,0.039314,-0.039230,0.0,160,8,20,0.800
6,Borderline,oldpeak > 0.48,0.036639,0.000830,0.036639,0.036609,1.0,20,1,20,0.100
7,Borderline,-0.94 < thal <= 1.11,0.035500,0.001141,0.035500,0.035514,1.0,80,4,20,0.400
8,Borderline,thal <= -0.94,-0.035314,0.001202,0.035314,-0.035284,0.0,120,6,20,0.600
9,Borderline,ca <= -0.73,-0.024313,0.001210,0.024313,-0.024319,0.0,140,7,20,0.700


LIME seed-stability summary:


,MeanPairwiseSpearman,MeanTopKOverlap,MeanRuleWeightSD,MeanRuleSignConsistency
CaseType,,,,
Borderline,0.991,0.978,0.002,0.917
FN,0.993,0.981,0.002,0.900
FP,0.974,0.953,0.003,0.917
TN,0.988,0.990,0.002,0.900
TP,0.866,0.867,0.005,0.867


### Bootstrap-refit stability


In [17]:
if stability_results is not None:
    display(stability_results["raw"].head())
    print(stability_results["summary"])


,Feature,Importance,ImportanceStd,Bootstrap,BootstrapSampleSeed,Rank,InTopK
0,age,0.000260,0.000565,1,383329927,10.0,False
1,sex,0.055747,0.033703,1,383329927,2.0,True
2,cp,0.048989,0.026177,1,383329927,3.0,True
3,trestbps,0.030572,0.023246,1,383329927,6.0,False
4,chol,-0.005311,0.003363,1,383329927,11.0,False


     Feature  MeanImportance  StdImportance  MeanRank   StdRank  TopKFrequency
1         ca        0.114938       0.101963      1.86  1.463495           0.91
10      thal        0.056352       0.033974      2.57  2.075616           0.77
3         cp        0.048993       0.045115      3.48  1.351991           0.63
8        sex        0.039026       0.048195      5.25  2.417842           0.27
11   thalach        0.024715       0.028456      5.61  2.658681           0.21
4      exang        0.014500       0.012938      6.67  2.287670           0.05
9      slope        0.021678       0.035659      7.17  2.322725           0.05
12  trestbps        0.020447       0.024348      7.34  2.483155           0.07
6    oldpeak        0.011827       0.017812      7.73  2.877481           0.04
2       chol        0.006647       0.014685      9.93  2.060965           0.00
0        age        0.002752       0.006042     10.11  1.391859           0.00
7    restecg        0.001383       0.004102     10.5

### Exported artefacts


In [18]:

out_dir = Path(config["output_dir"]).resolve()

print("Output directory:", out_dir)

for path in sorted(out_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(out_dir))


Output directory: C:\Users\catia\Desktop\Blackboard\AASB\github\thesis\results\heart
bundle\metadata.json
bundle\model.pkl
bundle\X_test.csv
bundle\X_train.csv
bundle\y_test.csv
bundle\y_train.csv
execution_times.json
lime_explanations.csv
lime_seed_stability.csv
lime_summary.csv
model.pkl
native_importance.csv
permutation_importance.csv
rank_agreement\Native_vs_Permutation.json
rank_agreement\Native_vs_SHAP.json
rank_agreement\SHAP_vs_Permutation.json
rank_compare.csv
representative_cases.csv
shap_importance.csv
stability.csv
stability_rank.csv
stability_summary.csv


In [19]:
print(
    experiment.state.results["local"].keys()
)


dict_keys(['cases', 'lime', 'lime_summary', 'lime_seed_stability'])


In [20]:
print(
    experiment.state.results["local"]["cases"].columns.tolist()
)

display(
    experiment.state.results["local"]["cases"]
)


['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'y_true', 'y_pred', 'y_prob', 'distance', 'CaseType', 'CaseIndex', 'CaseRank']


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,y_true,y_pred,y_prob,distance,CaseType,CaseIndex,CaseRank
84,-0.411115,0.619013,0.847174,0.506570,0.853934,-0.428952,-0.957016,-1.484606,1.421411,2.757194,0.703526,2.659046,1.114241,1,1,0.762068,0.262068,TP,84,1
2,0.140017,-1.615474,0.847174,4.062575,0.678110,2.331262,1.052039,-0.952943,1.421411,2.582322,2.356255,1.528188,1.114241,1,1,0.756083,0.256083,TP,2,2
125,0.360470,0.619013,0.847174,-0.204630,-0.587822,-0.428952,1.052039,-1.049609,1.421411,1.008470,0.703526,2.659046,1.114241,1,1,0.745467,0.245467,TP,125,3
14,0.360470,0.619013,0.847174,-0.204630,0.168221,-0.428952,1.052039,-1.097942,1.421411,1.707960,0.703526,1.528188,1.114241,1,1,0.736840,0.236840,TP,14,4
46,1.352509,0.619013,0.847174,-0.678764,-0.359251,-0.428952,1.052039,-1.146275,1.421411,1.358215,0.703526,1.528188,1.114241,1,1,0.735711,0.235711,TP,46,5
26,-0.190662,0.619013,0.847174,-0.500964,0.572616,-0.428952,-0.957016,-2.789597,1.421411,0.833598,0.703526,1.528188,1.114241,1,1,0.728936,0.228936,TP,26,6
18,0.360470,0.619013,0.847174,-1.034365,1.205582,-0.428952,0.047511,-0.614613,-0.703526,2.932067,2.356255,2.659046,0.600510,1,1,0.723356,0.223356,TP,18,7
74,1.352509,0.619013,0.847174,1.691905,0.642946,-0.428952,1.052039,-2.161268,1.421411,0.396417,0.703526,2.659046,-0.940683,1,1,0.717047,0.217047,TP,74,8
11,0.250244,0.619013,0.847174,1.217771,0.431957,-0.428952,-0.957016,-3.127927,1.421411,0.134108,0.703526,0.397329,1.114241,1,1,0.716343,0.216343,TP,11,9
60,1.021829,0.619013,0.847174,-0.678764,-0.060350,-0.428952,1.052039,-2.741264,1.421411,1.008470,2.356255,0.397329,-0.940683,1,1,0.707111,0.207111,TP,60,10


In [21]:
assert "CaseIndex" in experiment.state.results["local"]["cases"].columns

print("CaseIndex exists in memory.")


CaseIndex exists in memory.


In [22]:
heart_cases = pd.read_csv(
    Path("../../results/heart/representative_cases.csv")
)

print(heart_cases.columns.tolist())
display(heart_cases)


['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'y_true', 'y_pred', 'y_prob', 'distance', 'CaseType', 'CaseIndex', 'CaseRank']


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,y_true,y_pred,y_prob,distance,CaseType,CaseIndex,CaseRank
0,-0.411115,0.619013,0.847174,0.506570,0.853934,-0.428952,-0.957016,-1.484606,1.421411,2.757194,0.703526,2.659046,1.114241,1,1,0.762068,0.262068,TP,84,1
1,0.140017,-1.615474,0.847174,4.062575,0.678110,2.331262,1.052039,-0.952943,1.421411,2.582322,2.356255,1.528188,1.114241,1,1,0.756083,0.256083,TP,2,2
2,0.360470,0.619013,0.847174,-0.204630,-0.587822,-0.428952,1.052039,-1.049609,1.421411,1.008470,0.703526,2.659046,1.114241,1,1,0.745467,0.245467,TP,125,3
3,0.360470,0.619013,0.847174,-0.204630,0.168221,-0.428952,1.052039,-1.097942,1.421411,1.707960,0.703526,1.528188,1.114241,1,1,0.736840,0.236840,TP,14,4
4,1.352509,0.619013,0.847174,-0.678764,-0.359251,-0.428952,1.052039,-1.146275,1.421411,1.358215,0.703526,1.528188,1.114241,1,1,0.735711,0.235711,TP,46,5
5,-0.190662,0.619013,0.847174,-0.500964,0.572616,-0.428952,-0.957016,-2.789597,1.421411,0.833598,0.703526,1.528188,1.114241,1,1,0.728936,0.228936,TP,26,6
6,0.360470,0.619013,0.847174,-1.034365,1.205582,-0.428952,0.047511,-0.614613,-0.703526,2.932067,2.356255,2.659046,0.600510,1,1,0.723356,0.223356,TP,18,7
7,1.352509,0.619013,0.847174,1.691905,0.642946,-0.428952,1.052039,-2.161268,1.421411,0.396417,0.703526,2.659046,-0.940683,1,1,0.717047,0.217047,TP,74,8
8,0.250244,0.619013,0.847174,1.217771,0.431957,-0.428952,-0.957016,-3.127927,1.421411,0.134108,0.703526,0.397329,1.114241,1,1,0.716343,0.216343,TP,11,9
9,1.021829,0.619013,0.847174,-0.678764,-0.060350,-0.428952,1.052039,-2.741264,1.421411,1.008470,2.356255,0.397329,-0.940683,1,1,0.707111,0.207111,TP,60,10


In [23]:

print("Python:", sys.executable)
print("Python version:", sys.version)
print("scikit-learn:", sklearn.__version__)
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("SHAP:", shap.__version__)
print("statsmodels:", statsmodels.__version__)


Python: c:\Users\catia\AppData\Local\Programs\Python\Python312\python.exe
Python version: 3.12.6 (tags/v3.12.6:a4a2d2b, Sep  6 2024, 20:11:23) [MSC v.1940 64 bit (AMD64)]
scikit-learn: 1.6.1
NumPy: 2.1.3
pandas: 2.2.3
SHAP: 0.51.0
statsmodels: 0.14.6


In [24]:

results_dir = Path(config["output_dir"]).resolve()
stability_file = results_dir / "stability.csv"

print("Reading:", stability_file)

stability = pd.read_csv(stability_file)

importance_by_bootstrap = stability.pivot(
    index="Bootstrap",
    columns="Feature",
    values="Importance",
)

print("Bootstrap refits:", len(importance_by_bootstrap))
print(
    "Distinct importance profiles:",
    len(importance_by_bootstrap.drop_duplicates()),
)

display(importance_by_bootstrap)


Reading: C:\Users\catia\Desktop\Blackboard\AASB\github\thesis\results\heart\stability.csv
Bootstrap refits: 100
Distinct importance profiles: 100


Feature,age,ca,chol,cp,exang,fbs,oldpeak,restecg,sex,slope,thal,thalach,trestbps
Bootstrap,,,,,,,,,,,,,
1,0.000260,0.270941,-0.005311,0.048989,0.043259,-0.014671,4.138852e-03,-0.007095,0.055747,0.020064,0.038402,0.008029,0.030572
2,0.000373,0.197990,-0.000772,0.031926,0.022405,-0.021779,-7.178613e-05,-0.000312,0.060177,0.025888,0.066747,0.031463,0.025836
3,0.002344,0.027030,-0.002915,0.027849,0.015103,-0.002302,1.548694e-02,0.000853,0.009587,0.008056,0.030947,0.017619,0.003510
4,0.002228,0.051200,-0.000236,0.037085,0.017982,-0.007768,1.356621e-02,0.002763,0.018980,0.015028,0.077263,0.024101,-0.000051
5,0.002744,0.154101,0.004617,0.046523,0.008078,-0.005423,9.899828e-03,-0.004647,0.062494,-0.003212,0.040220,0.097215,0.089169
...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,-0.000225,0.080214,-0.003193,0.029974,0.024103,-0.005536,1.953515e-02,0.001957,0.014929,0.008274,0.056093,0.028676,0.011526
97,0.000406,0.144927,0.012339,0.051696,0.004633,-0.005411,-7.005550e-07,0.002621,0.026045,0.034407,0.099312,0.003402,0.010004
98,0.004578,0.019756,0.001320,0.024496,0.020889,-0.001989,1.560114e-02,0.002324,0.001745,0.009165,0.028469,0.014858,0.002456


## Interpretation

Native coefficient magnitude and SHAP produced closely aligned global rankings, whereas permutation importance diverged more often. Bootstrap-refit rank persistence was positive but was the lowest of the four retrainable configurations. Repeated-seed LIME results were stable for the deliberately selected cases, but local consistency does not establish prediction correctness or clinical validity.
